# Notebook 9 — Exploratory Data Analysis: exercise solutions

Solutions to the open exercises in `notebooks/09_exploratory_data_analysis.ipynb`
(Advanced Data Visualization for AI, Freie Universität Berlin).

The course notebook works through correlation matrices, parallel coordinates
charts and scatter plot matrices on the sklearn *wine* dataset, and leaves nine
exercises open. This notebook solves the eight that use the built-in datasets;
Homework 8 (`ori.dat`) is a separate notebook.

| # | Difficulty | Exercise |
|---|---|---|
| 1 | easy | Show only the upper triangle of the correlation matrix |
| 2 | medium | Pairwise mutual information matrix |
| 3 | easy | Reorder the parallel-coordinates axes |
| 4 | easy | Normalise instead of standardise |
| 5 | medium | Hover a line to grey out the other classes |
| 6 | hard | Min / median / max per variable |
| 7 | easy | Linking and brushing on the scatter matrix |
| 8 | hard | Lower triangle only of the scatter matrix |


## Setup

In [1]:
import numpy as np, pandas as pd, altair as alt
from sklearn import datasets
from sklearn.metrics import mutual_info_score as mis
from sklearn.preprocessing import StandardScaler, MinMaxScaler

data_wine = datasets.load_wine(as_frame=True).frame
FEATURES = [c for c in data_wine.columns if c != 'target']

## Exercise 1 (easy) — upper triangle of correlation matrix

A correlation matrix is symmetric, so the lower half repeats what the upper half already says. Dropping it halves the ink without losing information. The features are kept in their original order via `sort=`, otherwise Altair would sort the axes alphabetically and the triangle would break apart.

In [2]:
def corr_triangle(data, which='upper'):
    new = data.drop('target', axis=1)
    corr = new.corr(method='pearson')
    order = list(corr.columns)
    idx = {v: i for i, v in enumerate(order)}
    corr = corr.reset_index().melt('index')
    corr.columns = ['var_1', 'var_2', 'correlation']
    i1 = corr['var_1'].map(idx); i2 = corr['var_2'].map(idx)
    # y is drawn top-down, so the upper-right triangle is index(var_1) >= index(var_2)
    corr = corr[i1 >= i2] if which == 'upper' else corr[i1 <= i2]
    return alt.Chart(corr).mark_rect().encode(
        alt.X('var_1', title=None, sort=order, axis=alt.Axis(labelAngle=-45)),
        alt.Y('var_2', title=None, sort=order),
        alt.Color('correlation:Q', scale=alt.Scale(scheme='blueorange', domain=[-1, 1])),
        tooltip=['var_1', 'var_2', alt.Tooltip('correlation:Q', format='.2f')],
    ).properties(width=400, height=400, title='Pearson correlation (upper triangle)')
c1 = corr_triangle(data_wine, 'upper')

c1

alt.Chart(...)

## Exercise 2 (medium) — pairwise mutual information matrix

Mutual information measures any dependence between two variables, not just the linear kind Pearson correlation picks up. `sklearn`'s `mutual_info_score` expects discrete labels, so each continuous feature is binned into 10 equal-width bins first. The diagonal is each feature's own entropy, which is why it dominates the colour scale.

In [3]:
def mutual_info_matrix(data, bins=10):
    """MI needs discrete variables, so continuous features are binned first."""
    new = data.drop('target', axis=1)
    disc = new.apply(lambda s: pd.cut(s, bins=bins, labels=False))
    cols = list(disc.columns)
    m = pd.DataFrame(
        [[mis(disc[a], disc[b]) for b in cols] for a in cols],
        index=cols, columns=cols)
    return m, cols

def mi_chart(data, bins=10):
    m, order = mutual_info_matrix(data, bins)
    long = m.reset_index().melt('index')
    long.columns = ['var_1', 'var_2', 'mutual_information']
    return alt.Chart(long).mark_rect().encode(
        alt.X('var_1', title=None, sort=order, axis=alt.Axis(labelAngle=-45)),
        alt.Y('var_2', title=None, sort=order),
        alt.Color('mutual_information:Q', scale=alt.Scale(scheme='viridis')),
        tooltip=['var_1', 'var_2', alt.Tooltip('mutual_information:Q', format='.3f')],
    ).properties(width=400, height=400, title='Pairwise mutual information (10 bins)')
c2 = mi_chart(data_wine)

c2

alt.Chart(...)

## Exercise 3 (easy) — axis ordering on parallel coordinates

Axis order decides how many lines cross. Rather than shuffling by hand, the axes are ordered by an ANOVA-style ratio — between-class variance over total variance — so the features that separate the three wine cultivars best sit leftmost and the class bands are visible immediately.

In [4]:
def parallel_chart(data, order=None, title=''):
    long = data.reset_index().melt(id_vars=['index', 'target'])
    enc_x = alt.X('variable:N', sort=order, title=None, axis=alt.Axis(labelAngle=-45))
    return alt.Chart(long).mark_line().encode(
        enc_x, alt.Y('value:Q'), alt.Color('target:N'),
        alt.Detail('index:N'), opacity=alt.value(0.35),
    ).properties(width=700, height=300, title=title)

scaler = StandardScaler()
scaled = pd.DataFrame(scaler.fit_transform(data_wine[FEATURES]), columns=FEATURES)
scaled['target'] = data_wine['target']
sub = [c for c in FEATURES if c not in ('proline', 'magnesium')]

# order axes by how well each feature separates the classes (ANOVA-style ratio)
def separation_order(df, feats):
    score = {}
    for f in feats:
        g = df.groupby('target')[f]
        score[f] = g.mean().var() / (df[f].var() + 1e-12)
    return [f for f, _ in sorted(score.items(), key=lambda kv: -kv[1])]

ordered = separation_order(scaled, sub)
c3 = parallel_chart(scaled[sub + ['target']], order=ordered,
                    title='Parallel coordinates, axes ordered by class separation')

c3

alt.Chart(...)

## Exercise 4 (easy) — normalize (min-max) instead of standardize

Standardising centres each feature at zero with unit variance; min-max normalising squeezes every feature into [0, 1]. For parallel coordinates the second is often easier to read, because every axis then spans exactly the same range and the vertical position of a line is directly comparable across axes.

In [5]:
mm = MinMaxScaler()
normed = pd.DataFrame(mm.fit_transform(data_wine[FEATURES]), columns=FEATURES)
normed['target'] = data_wine['target']
c4 = parallel_chart(normed[sub + ['target']], order=ordered,
                    title='Parallel coordinates on min-max normalised data')

c4

alt.Chart(...)

## Exercise 5 (medium) — hover one class, gray out the others

`selection_point` with `on='mouseover'` and `fields=['target']` binds the hover to the whole class rather than the single line under the cursor. `alt.condition` then drives both colour and opacity, so the hovered class keeps its colour and everything else fades to grey.

In [6]:
long5 = normed[sub + ['target']].reset_index().melt(id_vars=['index', 'target'])
hover = alt.selection_point(fields=['target'], on='mouseover', empty=True)
c5 = alt.Chart(long5).mark_line().encode(
    alt.X('variable:N', sort=ordered, title=None, axis=alt.Axis(labelAngle=-45)),
    alt.Y('value:Q'), alt.Detail('index:N'),
    color=alt.condition(hover, alt.Color('target:N'), alt.value('lightgray')),
    opacity=alt.condition(hover, alt.value(0.7), alt.value(0.1)),
).add_params(hover).properties(
    width=700, height=300, title='Hover a line to highlight its class')

c5

alt.Chart(...)

## Exercise 6 (hard) — min / median / max per variable

The summary statistics are computed with `agg` and drawn in two layers: a shaded area between min and max to show each feature's range, and three lines for min, median and max. Layering the band under the lines keeps the median readable where the range is wide.

In [7]:
stats = scaled[sub].agg(['min', 'median', 'max']).T.reset_index()
stats.columns = ['variable', 'min', 'median', 'max']
stats_long = stats.melt('variable', var_name='statistic', value_name='value')
band = alt.Chart(stats).mark_area(opacity=0.25).encode(
    alt.X('variable:N', sort=ordered, title=None, axis=alt.Axis(labelAngle=-45)),
    alt.Y('min:Q', title='value'), alt.Y2('max:Q'))
lines = alt.Chart(stats_long).mark_line(point=True).encode(
    alt.X('variable:N', sort=ordered, title=None),
    alt.Y('value:Q'), alt.Color('statistic:N'),
    tooltip=['variable', 'statistic', alt.Tooltip('value:Q', format='.3f')])
c6 = (band + lines).properties(
    width=700, height=300, title='Min / median / max per variable (standardised)')

c6

alt.LayerChart(...)

## Exercise 7 (easy) — linking and brushing on the scatter matrix

A single `selection_interval` with `resolve='global'` is shared by every cell of the repeated chart. Dragging a box in any panel selects those data points everywhere at once — brushing in one panel, linked highlighting in the rest.

In [8]:
brush = alt.selection_interval(resolve='global')
small = FEATURES[:4]
c7 = alt.Chart(data_wine).mark_circle(size=40).encode(
    alt.X(alt.repeat('column'), type='quantitative', scale=alt.Scale(nice=True)),
    alt.Y(alt.repeat('row'), type='quantitative', scale=alt.Scale(nice=True)),
    color=alt.condition(brush, 'target:N', alt.value('lightgray')),
).add_params(brush).properties(width=140, height=140).repeat(
    row=small, column=small).properties(
    title='Scatter matrix with linking and brushing')

c7

alt.RepeatChart(...)

## Exercise 8 (hard) — lower triangle only of the scatter matrix

`repeat()` always produces a full grid and gives no way to skip cells, so the lower triangle has to be built explicitly: one `hconcat` per row containing only the cells at or left of the diagonal, all stacked with `vconcat`. Axis titles are drawn only on the bottom row and the left column to avoid repeating them in every panel.

In [9]:
def scatter_lower(data, feats, sel=None):
    """repeat() cannot skip cells, so the lower triangle is built by hconcat/vconcat."""
    rows = []
    for i, r in enumerate(feats):
        cells = []
        for j, c in enumerate(feats):
            if j > i:
                continue
            ch = alt.Chart(data).mark_circle(size=35).encode(
                alt.X(c, type='quantitative', scale=alt.Scale(nice=True),
                      axis=alt.Axis(title=c if i == len(feats) - 1 else None)),
                alt.Y(r, type='quantitative', scale=alt.Scale(nice=True),
                      axis=alt.Axis(title=r if j == 0 else None)),
                color=(alt.condition(sel, 'target:N', alt.value('lightgray'))
                       if sel is not None else alt.Color('target:N')),
            ).properties(width=130, height=130)
            if sel is not None:
                ch = ch.add_params(sel)
            cells.append(ch)
        rows.append(alt.hconcat(*cells))
    return alt.vconcat(*rows).properties(
        title='Scatter matrix, lower triangle only')
c8 = scatter_lower(data_wine, small, alt.selection_interval(resolve='global'))

c8

alt.VConcatChart(...)